In [299]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split


In [300]:
train = np.load("train.npz")
test = np.load("test.npz")

In [301]:
train_data = train["data"]
train_labels = train["labels"]
test_data = test["data"]

In [302]:
print("Full train data:", train_data.shape)
print("Full train labels:", train_labels.shape)
print("Test data:", test_data.shape)

Full train data: (60000, 784)
Full train labels: (60000,)
Test data: (15000, 784)


In [303]:
# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    train_data,
    train_labels,
    test_size=0.2,
    random_state=1,
    stratify=train_labels,
)

In [304]:
print("Training split:", X_train.shape)
print("Validation split:", X_val.shape)

Training split: (48000, 784)
Validation split: (12000, 784)


In [305]:
def one_hot(y, num_classes=10):
    mat = np.eye(num_classes)[np.array(y).reshape(-1)]
    return mat

y = [0, 2, 1]
nb_classes = 3
print(one_hot(y, nb_classes))

[[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]]


In [306]:
def softmax(z):
    z = z - np.max(z, axis=-1, keepdims=True)
    
    expsum = np.sum(np.exp(z), axis=-1, keepdims=True)
    return np.exp(z) / expsum

z = np.array([1.0, 2.0, 3.0])
print(softmax(z), np.sum(softmax(z)))

z = np.array([1000.0, 1001.0, 1002.0])
print(softmax(z), np.sum(softmax(z))) # hits overflow, needs z - max(z)

z = np.array([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])
print(softmax(z), np.sum(softmax(z), axis=-1, keepdims=True)) # hits size mismatch, needs keepdims

[0.09003057 0.24472847 0.66524096] 0.9999999999999999
[0.09003057 0.24472847 0.66524096] 0.9999999999999999
[[0.09003057 0.24472847 0.66524096]
 [0.09003057 0.24472847 0.66524096]] [[1.]
 [1.]]


In [307]:
def forward(X, W, b):
    # prediction
    z = X @ W + b

    # activation
    y_hat = softmax(z)

    return y_hat

# X.shape = (48000, 784)
# W.shape = (784, 10)
# b.shape = (10, )
X = np.random.randn(48000, 784)
W = np.random.randn(784, 10) * np.sqrt(2 / 784) # He initialization
b = np.zeros(10)
print(forward(X, W, b)[:2])

[[0.01593257 0.00429548 0.40501935 0.01344317 0.30376278 0.06605158
  0.09442853 0.0261401  0.07008015 0.0008463 ]
 [0.16480434 0.13929083 0.00224308 0.03520698 0.02103163 0.30279252
  0.08537641 0.06158698 0.01668873 0.1709785 ]]


In [308]:
def cross_entropy(y_hat, y, eps=1e-15):
    m = y_hat.shape[0]
    y_hat = np.clip(y_hat, eps, 1 - eps)
    return -1/m * np.sum(y * np.log(y_hat))

y = one_hot([0, 2], num_classes=3)
y_hat = np.array([
    [0.7, 0.2, 0.1],  
    [0.1, 0.1, 0.8]   
])
print(cross_entropy(y_hat, y))

y_hat = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0]
])
print(cross_entropy(y_hat, y)) # hits division by log 0, needs clipping

0.2899092476264711
9.992007221626415e-16


In [309]:
def backward(X, y_hat, y, W, b, vW, vb, lr, momentum=0.9):
    m = y_hat.shape[0]

    # dout = dL/dy_hat * dy_hat/dz
    dout = (y_hat - y) / m

    # dW = dL/dy_hat * dy_hat/dz * dz/dW = dout * X
    # X.shape = (48000, 748)
    # dout.shape = (48000, 10)
    dW = X.T @ dout

    # sum of dout across samples
    db = np.sum(dout, axis=0)

    # gradient descent
    vW = momentum * vW - lr * dW
    vb = momentum * vb - lr * db
    W += vW
    b += vb

    return W, b, vW, vb

X = np.array([[1.0, 2.0], [3.0, 4.0]])
y = one_hot([0, 1], num_classes=2)
W = np.zeros((2, 2))
b = np.zeros(2)
y_hat = np.array([[0.8, 0.2], [0.1, 0.9]])

#W_new, b_new = backward(X, y_hat, y, W, b, lr=0.1)
#print(W_new, b_new)

In [310]:
def predict(X, W, b):
    y_hat = forward(X, W, b)
    winner = np.argmax(y_hat, axis=-1)

    return winner

X = np.array([[1.0, 0.0], [0.0, 1.0]])
W = np.array([[10.0, -10.0], [-10.0, 10.0]])
b = np.zeros(2)
print(predict(X, W, b))

print(predict(X[0], W, b)) 

[0 1]
0


In [311]:
# setup
# X_train_norm = X_train / 255.0 if X_train.max() > 1.0 else X_train
# X_val_norm = X_val / 255.0 if X_val.max() > 1.0 else X_val
X_norm = train_data / 255.0
X_test_norm = test_data / 255.0

mean_px = np.mean(X_norm, axis=0)

X_norm = X_norm - mean_px
X_test_norm = X_test_norm - mean_px

# y_train_oh = one_hot(y_train, num_classes=10)
# y_val_oh = one_hot(y_val, num_classes=10)
y_oh = one_hot(train_labels, num_classes=10)

np.random.seed(42)
W = np.random.randn(784, 10) * np.sqrt(2 / 784) # He init
b = np.zeros(10)

# momentum
vW = np.zeros_like(W)
vb = np.zeros_like(b)
momentum = 0.9

# hyperparameters
epochs = 70
batch_size = 128
lr = 0.05
m = X_norm.shape[0]

In [312]:
# mini batch gsd
for epoch in range(epochs):
    indices = np.random.permutation(m)
    X_shuf = X_norm[indices]
    y_shuf = y_oh[indices]

    # loop over batches
    for i in range(0, m, batch_size):
        X_batch = X_shuf[i:i+batch_size]
        y_batch = y_shuf[i:i+batch_size]

        y_hat = forward(X_batch, W, b)
        W, b, vW, vb = backward(X_batch, y_hat, y_batch, W, b, vW, vb, lr, momentum)

    # log progrees
    if (epoch + 1) % 5 == 0 or epoch == 0:
        probs = forward(X_norm, W, b)
        loss = cross_entropy(probs, y_oh)
        acc = np.mean(predict(X_norm, W, b) == train_labels)
        print(f"Epoch {epoch + 1}/{epochs} | Train Loss: {loss:.4f} | Train Acc: {acc:.4f}")    

    lr *= 0.98

Epoch 1/70 | Train Loss: 0.3151 | Train Acc: 0.9101
Epoch 5/70 | Train Loss: 0.2701 | Train Acc: 0.9245
Epoch 10/70 | Train Loss: 0.2566 | Train Acc: 0.9289
Epoch 15/70 | Train Loss: 0.2511 | Train Acc: 0.9309
Epoch 20/70 | Train Loss: 0.2470 | Train Acc: 0.9324
Epoch 25/70 | Train Loss: 0.2445 | Train Acc: 0.9327
Epoch 30/70 | Train Loss: 0.2422 | Train Acc: 0.9335
Epoch 35/70 | Train Loss: 0.2405 | Train Acc: 0.9342
Epoch 40/70 | Train Loss: 0.2393 | Train Acc: 0.9345
Epoch 45/70 | Train Loss: 0.2383 | Train Acc: 0.9343
Epoch 50/70 | Train Loss: 0.2376 | Train Acc: 0.9344
Epoch 55/70 | Train Loss: 0.2367 | Train Acc: 0.9350
Epoch 60/70 | Train Loss: 0.2360 | Train Acc: 0.9352
Epoch 65/70 | Train Loss: 0.2355 | Train Acc: 0.9355
Epoch 70/70 | Train Loss: 0.2352 | Train Acc: 0.9356


In [316]:
train_accuracy = np.mean(predict(X_norm, W, b) == train_labels)
print(f"Training accuracy: {train_accuracy:.4f}")
predictions = predict(X_test_norm, W, b)

Training accuracy: 0.9356


In [317]:
submission = pd.DataFrame({
    "ID": range(len(predictions)),
    "target": predictions,
})

In [318]:
submission.to_csv("submission.csv", index=False)
print("Created submission.csv")

Created submission.csv
